# AlzDetect — v4: Real Fuzzy Inference Head

Up to v3, the only "fuzzy" part was the skfuzzy **resampling controller** (it just picks per-class target counts). The classifier itself was a plain Dense head.

This notebook keeps the winning v3 recipe (medical-safe aug + frozen MobileNetV2 features + SMOTE) but replaces the Dense head with a **trainable Takagi–Sugeno–Kang fuzzy inference layer**:

`features → Dense(8, relu) projection → FuzzyLayer(Gaussian membership → fuzzy rules → defuzzification) → class probs`

The `FuzzyLayer` self-registers, so the exported `alz_mobilenetv2.keras` is a drop-in for the backend (which imports the identical layer).

**Runtime → GPU.**

In [ ]:
!pip install -q kaggle scikit-fuzzy imbalanced-learn opencv-python-headless

## Kaggle credentials + dataset

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = ''  # <-- your kaggle username
os.environ['KAGGLE_KEY']      = ''  # <-- your kaggle API key
assert os.environ['KAGGLE_USERNAME'] and os.environ['KAGGLE_KEY'], 'Fill in Kaggle credentials.'
!kaggle datasets download -d legendahmed/alzheimermridataset -p data --unzip
DATA_DIR = 'data/Alzheimer_s Dataset/all image'
print('Files found:', len(os.listdir(DATA_DIR)))

## The fuzzy inference layer (TSK / ANFIS-style)
Gaussian membership functions → fuzzy rule firing (fuzzy AND) → normalized defuzzification. Centers, widths and consequents are all learned by backprop. Must match `backend/fuzzy_layer.py` exactly (same `package`/`name`) so the saved model loads in the app.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

@tf.keras.utils.register_keras_serializable(package='alzdetect', name='FuzzyLayer')
class FuzzyLayer(layers.Layer):
    def __init__(self, n_rules, n_classes, **kwargs):
        super().__init__(**kwargs)
        self.n_rules = int(n_rules)
        self.n_classes = int(n_classes)
    def build(self, input_shape):
        d = int(input_shape[-1])
        self.centers = self.add_weight(name='centers', shape=(self.n_rules, d),
            initializer=tf.keras.initializers.RandomNormal(stddev=1.0), trainable=True)
        self.log_sigma = self.add_weight(name='log_sigma', shape=(self.n_rules, d),
            initializer=tf.keras.initializers.Zeros(), trainable=True)
        self.consequent = self.add_weight(name='consequent', shape=(self.n_rules, self.n_classes),
            initializer='glorot_uniform', trainable=True)
        super().build(input_shape)
    def call(self, inputs):
        x = tf.expand_dims(inputs, axis=1)                 # (B,1,D)
        c = tf.expand_dims(self.centers, axis=0)           # (1,R,D)
        sigma = tf.exp(tf.expand_dims(self.log_sigma, 0))  # (1,R,D) > 0
        log_mu = -0.5 * tf.reduce_sum(tf.square((x - c) / sigma), axis=-1)  # (B,R)
        firing = tf.nn.softmax(log_mu, axis=1)             # normalized firing strengths
        out = tf.matmul(firing, self.consequent)           # (B,n_classes)
        return tf.nn.softmax(out)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'n_rules': self.n_rules, 'n_classes': self.n_classes})
        return cfg

## Config + helpers (fuzzy resampling controller, medical-safe aug)

In [ ]:
import re
import numpy as np

IMG_SIZE, EPOCHS, BATCH, SEED = 128, 60, 32, 42
PROJ_DIM, N_RULES = 8, 16
np.random.seed(SEED)

PREFIX_TO_CLASS = {'mildDem':'Mild Dementia','moderateDem':'Moderate Dementia',
                   'verymildDem':'Very mild Dementia','nonDem':'Non Demented'}
CLASS_ORDER = ['Mild Dementia','Moderate Dementia','Very mild Dementia','Non Demented']

def fuzzy_targets(counts):
    import skfuzzy as fuzz
    import skfuzzy.control as ctrl
    imb = ctrl.Antecedent(np.arange(0,1.1,0.1),'imbalance')
    res = ctrl.Consequent(np.arange(0,1.1,0.1),'resample_factor')
    imb['low']=fuzz.trimf(imb.universe,[0,0,0.5]); imb['medium']=fuzz.trimf(imb.universe,[0.3,0.5,0.7]); imb['high']=fuzz.trimf(imb.universe,[0.5,1,1])
    res['low']=fuzz.trimf(res.universe,[0,0,0.3]); res['medium']=fuzz.trimf(res.universe,[0.2,0.5,0.7]); res['high']=fuzz.trimf(res.universe,[0.6,1,1])
    sim = ctrl.ControlSystemSimulation(ctrl.ControlSystem([
        ctrl.Rule(imb['low'],res['low']), ctrl.Rule(imb['medium'],res['medium']), ctrl.Rule(imb['high'],res['high'])]))
    nmax=max(counts.values()); out={}
    for c,n in counts.items():
        sim.input['imbalance']=1-(n/nmax); sim.compute(); out[c]=int(n+sim.output['resample_factor']*nmax)
    return out

def bucket_files():
    b={c:[] for c in CLASS_ORDER}
    for fn in sorted(os.listdir(DATA_DIR)):
        m=re.match(r'^[a-zA-Z]+',fn)
        if m and PREFIX_TO_CLASS.get(m.group()): b[PREFIX_TO_CLASS[m.group()]].append(fn)
    return b

def medical_safe_aug(img,rng):
    import cv2
    angle=rng.uniform(-5,5); h,w=img.shape[:2]
    M=cv2.getRotationMatrix2D((w/2,h/2),angle,1.0)
    img=cv2.warpAffine(img,M,(w,h),borderMode=cv2.BORDER_REFLECT)
    if rng.random()<0.5: img=cv2.flip(img,1)
    return img

## Load images + medical-safe oversampling

In [ ]:
import cv2
rng = np.random.default_rng(SEED)
buckets = bucket_files()
counts = {c: len(f) for c, f in buckets.items()}
print('Real counts:', counts)
targets = fuzzy_targets(counts)
print('Fuzzy target counts:', targets)

class_map = {label: i for i, label in enumerate(CLASS_ORDER)}
data, labels = [], []
for label in CLASS_ORDER:
    files, target = buckets[label], targets[label]
    imgs = [cv2.resize(im, (IMG_SIZE, IMG_SIZE)) for im in
            (cv2.imread(os.path.join(DATA_DIR, fn)) for fn in files) if im is not None]
    for im in imgs:
        data.append(im); labels.append(class_map[label])
    i = 0
    while imgs and sum(1 for l in labels if l == class_map[label]) < target:
        data.append(medical_safe_aug(imgs[i % len(imgs)], rng)); labels.append(class_map[label]); i += 1

data = np.array(data, dtype=np.float32)
y_int = np.array(labels)
print('After safe aug:', {CLASS_ORDER[i]: int((y_int == i).sum()) for i in range(4)})

## Frozen MobileNetV2 features + SMOTE

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

Xtr_img, Xte_img, ytr_int, yte_int = train_test_split(
    data, y_int, test_size=0.2, stratify=y_int, random_state=SEED)

inp = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
base = MobileNetV2(weights='imagenet', include_top=False, input_tensor=inp)
base.trainable = False
feature_model = Model(inp, GlobalAveragePooling2D()(base.output))

print('Extracting frozen features ...')
Ftr = feature_model.predict(Xtr_img, batch_size=BATCH, verbose=1)
Fte = feature_model.predict(Xte_img, batch_size=BATCH, verbose=1)

min_class = min(int((ytr_int == i).sum()) for i in range(4))
k = max(1, min(5, min_class - 1))
print(f'SMOTE k_neighbors={k}')
Ftr_bal, ytr_bal = SMOTE(random_state=SEED, k_neighbors=k).fit_resample(Ftr, ytr_int)
print('Balanced:', {CLASS_ORDER[i]: int((ytr_bal == i).sum()) for i in range(4)})
ytr_cat = to_categorical(ytr_bal, 4)
yte_cat = to_categorical(yte_int, 4)

## Train the fuzzy inference head

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report

feat_in = Input(shape=(Ftr.shape[1],))
proj = Dense(PROJ_DIM, activation='relu', name='fuzzy_projection')(feat_in)
head_out = FuzzyLayer(n_rules=N_RULES, n_classes=4, name='fuzzy_head')(proj)
head = Model(feat_in, head_out)
head.compile(optimizer=Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
print(f'Fuzzy head: PROJ_DIM={PROJ_DIM}, N_RULES={N_RULES}')

head.fit(Ftr_bal, ytr_cat, validation_data=(Fte, yte_cat),
         epochs=EPOCHS, batch_size=BATCH,
         callbacks=[EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)])

yp = np.argmax(head.predict(Fte), axis=1)
print('\nClassification report:')
print(classification_report(yte_int, yp, target_names=CLASS_ORDER))

## Stitch into one drop-in model + download

In [ ]:
full = Model(inp, head(feature_model(inp)))
full.save('alz_mobilenetv2.keras')
print('Saved alz_mobilenetv2.keras (fuzzy head)')
from google.colab import files
files.download('alz_mobilenetv2.keras')